# Training base expert on vanilla OGBench environment using BC

In [ ]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
num_steps = 2000
seed = 0
hidden_dims = {'W'}

random.seed(seed)
torch.manual_seed(seed)

In [ ]:
env = HumanoidMazePCH(env_id='humanoidmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, custom_hidden=hidden_dims, expert_mode=True, seed=seed, success_radius=20.0)
train_eps = env.expert.num_eps
train_eps

In [ ]:
X = {f'X{t}' for t in range(num_steps)}
Y = f'Y{num_steps}'
obs_prefix = env.env.observed_unobserved_vars[0]

In [ ]:
Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')

    Z_sets[Xi] = cond

Z_sets['X1']

In [ ]:
records = collect_expert_trajectories(
    env,
    num_episodes=train_eps,
    max_steps=num_steps,
    seed=seed,
    show_progress=True
)

In [ ]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 20
lookback = 5
num_blocks = 4
epochs = 200
dropout = 0.0

dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    'V': 3,
    'C': 3,
    'J': 27,
    'X': 21
}

In [ ]:
model, slots, Z_trim = train_single_policy_long_horizon(
    records,
    Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions = env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(env.action_space.low, env.action_space.high)
)

policy = shared_policy_fn_long_horizon(model, slots, Z_trim, continuous=True, device=device)
policies = make_shared_policy_dict(policy)

In [ ]:
expert_episode_rewards = defaultdict(float)
for rec in records:
    ep = rec['episode']
    expert_episode_rewards[ep] += float(rec['reward'])

num_eps = len(expert_episode_rewards)
expert_rewards = [expert_episode_rewards[e] for e in range(num_eps)]

num_eval_eps = 10

policy_records = collect_imitator_trajectories(
    env=env,
    policies=policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

policy_episode_rewards = defaultdict(float)
for rec in policy_records:
    ep = rec['episode']
    policy_episode_rewards[ep] += float(rec['reward'])

policy_rewards = [policy_episode_rewards[e] for e in range(num_eval_eps)]

sum(expert_rewards)/num_eps, sum(policy_rewards)/num_eval_eps

In [ ]:
# save model for fine-tuning
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'humanoidmaze_large_expert.pt')

checkpoint = {
    "state_dict": model.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env.action_space.low,
    "action_bounds_high": env.action_space.high,
    "input_dim": int(model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)